<a href="https://colab.research.google.com/github/MohitKhetan10/Information-Retrieval/blob/week1/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Information Retrieval

In [1]:
from google.colab import files
import os

uploaded = files.upload()

folder_path = "/content/datasets"
os.makedirs(folder_path, exist_ok=True)

for filename in uploaded.keys():
    os.rename(filename, os.path.join(folder_path, filename))

print("Uploaded files:", list(uploaded.keys()))


Saving data_preparation.txt to data_preparation.txt
Saving evaluation.txt to evaluation.txt
Saving model_building.txt to model_building.txt
Saving conclusion.txt to conclusion.txt
Saving visualization.txt to visualization.txt
Uploaded files: ['data_preparation.txt', 'evaluation.txt', 'model_building.txt', 'conclusion.txt', 'visualization.txt']


In [4]:
import nltk, os, shutil

# Clean out any broken nltk_data folders
shutil.rmtree("/root/nltk_data", ignore_errors=True)
os.makedirs("/root/nltk_data", exist_ok=True)

# Force download punkt into that folder
nltk.download("punkt", download_dir="/root/nltk_data")
nltk.download("stopwords", download_dir="/root/nltk_data")
nltk.download("wordnet", download_dir="/root/nltk_data")

# Add that directory to NLTK's search path
nltk.data.path.clear()
nltk.data.path.append("/root/nltk_data")

# Verify punkt is available
print("Available:", nltk.data.find("tokenizers/punkt"))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Available: /root/nltk_data/tokenizers/punkt


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [6]:
import nltk, os

# Ensure nltk_data folder exists
os.makedirs("/root/nltk_data", exist_ok=True)

# Download both punkt and punkt_tab into that folder
nltk.download("punkt", download_dir="/root/nltk_data")
nltk.download("punkt_tab", download_dir="/root/nltk_data")
nltk.download("stopwords", download_dir="/root/nltk_data")
nltk.download("wordnet", download_dir="/root/nltk_data")

# Add folder to NLTK's search path
nltk.data.path.append("/root/nltk_data")

# Verify availability
print("punkt:", nltk.data.find("tokenizers/punkt"))
print("punkt_tab:", nltk.data.find("tokenizers/punkt_tab"))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


punkt: /root/nltk_data/tokenizers/punkt
punkt_tab: /root/nltk_data/tokenizers/punkt_tab


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
from nltk.tokenize import word_tokenize

print(word_tokenize("Neural Networks, and CART/C4.5 models!"))


['Neural', 'Networks', ',', 'and', 'CART/C4.5', 'models', '!']


In [8]:
def load_text_files(folder_path):
    data = {}
    doc_id_to_filename = {}
    for idx, filename in enumerate(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
                data[idx] = f.read()
            doc_id_to_filename[idx] = filename
    return data, doc_id_to_filename

# Load uploaded files
data, doc_id_to_filename = load_text_files("/content/datasets")
print("Loaded files:", list(doc_id_to_filename.values()))


Loaded files: ['model_building.txt', 'visualization.txt', 'conclusion.txt', 'evaluation.txt', 'data_preparation.txt']


In [9]:
from collections import defaultdict, Counter

def build_inverted_index(data):
    inverted_index = defaultdict(set)
    term_frequencies = Counter()
    for doc_id, content in data.items():
        cleaned_tokens = clean_text(content)
        for token in cleaned_tokens:
            inverted_index[token].add(doc_id)
            term_frequencies[token] += 1
    return inverted_index, term_frequencies

inverted_index, term_frequencies = build_inverted_index(data)
print("Top 5 terms:", term_frequencies.most_common(5))


Top 5 terms: [('model', 7), ('using', 6), ('c4', 4), ('5', 4), ('cost', 4)]


In [10]:
def boolean_and(terms, inverted_index):
    result_set = inverted_index.get(terms[0], set())
    for term in terms[1:]:
        result_set = result_set.intersection(inverted_index.get(term, set()))
    return result_set

def boolean_or(terms, inverted_index):
    result_set = set()
    for term in terms:
        result_set = result_set.union(inverted_index.get(term, set()))
    return result_set

def boolean_not(term, inverted_index, total_docs):
    return set(range(total_docs)) - inverted_index.get(term, set())

def boolean_query(query, inverted_index, total_docs):
    tokens = query.lower().split()

    if 'and' in tokens and 'not' in tokens:
        pos_terms = [t for t in tokens if t not in ['and','or','not']]
        not_term = tokens[tokens.index('not') + 1]
        base = boolean_and(pos_terms, inverted_index) if pos_terms else set(range(total_docs))
        return base.intersection(boolean_not(not_term, inverted_index, total_docs))

    if 'and' in tokens:
        terms = [t for t in tokens if t not in ['and','or','not']]
        return boolean_and(terms, inverted_index)

    if 'or' in tokens:
        terms = [t for t in tokens if t not in ['and','or','not']]
        return boolean_or(terms, inverted_index)

    if tokens and tokens[0] == 'not':
        return boolean_not(tokens[1], inverted_index, total_docs)

    return inverted_index.get(tokens[0], set())


In [11]:
def convert_doc_ids_to_filenames(result_set, doc_id_to_filename):
    return [doc_id_to_filename[doc_id] for doc_id in result_set if doc_id in doc_id_to_filename]

def write_query_results(queries, inverted_index, doc_id_to_filename, total_docs, out_path="query_results.txt"):
    with open(out_path, "w", encoding="utf-8") as result_file:
        for query in queries:
            result_ids = boolean_query(query, inverted_index, total_docs)
            result_files = convert_doc_ids_to_filenames(result_ids, doc_id_to_filename)
            result_str = f"Results for '{query}': {result_files}\n"
            print(result_str.strip())
            result_file.write(result_str)

queries = [
    "customer AND churn",
    "lift OR accuracy",
    "neural AND network",
    "C4.5 OR CART",
    "cost AND NOT sensitivity"
]

write_query_results(queries, inverted_index, doc_id_to_filename, len(data))

print("\nTop 10 frequent terms:")
for term, freq in term_frequencies.most_common(10):
    print(f"{term}: {freq}")


Results for 'customer AND churn': []
Results for 'lift OR accuracy': ['visualization.txt', 'conclusion.txt', 'evaluation.txt']
Results for 'neural AND network': ['model_building.txt', 'visualization.txt', 'conclusion.txt']
Results for 'C4.5 OR CART': ['model_building.txt', 'visualization.txt']
Results for 'cost AND NOT sensitivity': []

Top 10 frequent terms:
model: 7
using: 6
c4: 4
5: 4
cost: 4
lift: 4
chart: 4
built: 3
neural: 3
network: 3


In [12]:
from google.colab import files

# Download the query results file
files.download("query_results.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# New Section